In [1]:
%matplotlib inline
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import copy

from docx import Document
from docx.shared import Inches
from docx.enum.text import WD_PARAGRAPH_ALIGNMENT, WD_ALIGN_PARAGRAPH

from datetime import datetime
from tqdm import tqdm

import Flexivan_Prediction_Package

In [2]:
# Get filenames from folder
print('Getting PAST raw data filenames...')
# DATA_Folder = '/root/Flexivan/Flexivan/Daily prediction/DATA'
DATA_Folder = '/root/Flexivan/Flexivan/inference_data_Oct_to_Dec_2025'

FILENAMES = [f for f in os.listdir(DATA_Folder) if os.path.isfile(os.path.join(DATA_Folder, f))]
selected = "Latest_Test_"
selected2 = '_DETAILED'
FILENAMES = [f for f in FILENAMES if selected in f and selected2 not in f]
DATES = Flexivan_Prediction_Package.extract_datetimes_from_filenames(FILENAMES)
Filenames_DF = pd.DataFrame({
    "Filenames": FILENAMES,
    "Dates": DATES
})
Filenames_DF_Sorted = Filenames_DF.sort_values(by='Dates')
FILENAMES = list(Filenames_DF_Sorted['Filenames'])
DATES = list(Filenames_DF_Sorted['Dates'])

print(f'DONE - {len(FILENAMES)} Filenames were found')

#endregion

Getting PAST raw data filenames...
DONE - 71 Filenames were found


In [5]:
# Get the last appearance date for each lot

Sorting_Field='CHS Pickup Date'
Pickup_LOT_Field_Name = 'CHS Pickup Loc'
Return_LOT_Field_Name = 'CHS Return Loc'
Columns_2_Drop_From_Training = ['CHS ID', 'CTR Trip Id', 'CHS Return Dt', 'CHS Pickup Date', 'CTR pick Dt', 'CTR Return Dt']
Enumerated_Columns_LIST = ['CHS Pickup Loc', 'CHS Return Loc', 'CHS pickup MCO', 'CTR Trip MCO', 'O Customer', 'Customer', 'DC Loc', 'CTR Pickup Term', 'CTR Return Term', 
                           'pgkey', 'CTR Trip Loc Type Pattern', 'CTR Trip Pattern']
LAST_APPEARANCE_DATES = {}

for file_index, filename in enumerate(FILENAMES):
    print(f'Analyzing latest dates for file No.{file_index+1} out of {len(FILENAMES)}...', end='\r')
    File_Analysis_Results_OBJ = Flexivan_Prediction_Package.File_Analysis_Reults(f'{DATA_Folder}/{filename}', Sorting_Field, Columns_2_Drop_From_Training, Enumerated_Columns_LIST)

    DATA = File_Analysis_Results_OBJ.DATA

    for __, row in DATA.iterrows():
        PU_LOT = row['CHS Pickup Loc']
        RET_LOT = row['CHS Return Loc']
        RET_Date = row['CHS Return Dt']
        PU_Date = row['CHS Pickup Date']

        try:
            if LAST_APPEARANCE_DATES[f'PU_LOT_{PU_LOT}_Last_Date']<PU_Date:
                LAST_APPEARANCE_DATES[f'PU_LOT_{PU_LOT}_Last_Date'] = PU_Date
                LAST_APPEARANCE_DATES[f'PU_LOT_{PU_LOT}_Last_File_Index'] = file_index
        except:
            LAST_APPEARANCE_DATES[f'PU_LOT_{PU_LOT}_Last_Date'] = PU_Date

        try:
            if LAST_APPEARANCE_DATES[f'RET_LOT_{RET_LOT}_Last_Date']<RET_Date:
                LAST_APPEARANCE_DATES[f'RET_LOT_{RET_LOT}_Last_Date'] = RET_Date
                LAST_APPEARANCE_DATES[f'RET_LOT_{RET_LOT}_Last_File_Index'] = file_index
        except:
            LAST_APPEARANCE_DATES[f'RET_LOT_{RET_LOT}_Last_Date'] = RET_Date

print('DONE.')


DONE.zing latest dates for file No.71 out of 71...


In [7]:
LAST_APPEARANCE_DATES

{'PU_LOT_LAXCSN_Last_Date': Timestamp('2025-12-03 15:54:28'),
 'RET_LOT_LAXCSN_Last_Date': Timestamp('2025-12-10 02:46:25'),
 'PU_LOT_LAXAIM_Last_Date': Timestamp('2025-12-02 14:17:01'),
 'RET_LOT_LAXAIM_Last_Date': Timestamp('2025-12-09 15:55:31'),
 'PU_LOT_LAXCSN_Last_File_Index': 70,
 'PU_LOT_LAXAIM_Last_File_Index': 66,
 'RET_LOT_LAXAIM_Last_File_Index': 70,
 'RET_LOT_LAXCSN_Last_File_Index': 70,
 'RET_LOT_LAXEMS_Last_Date': Timestamp('2025-10-01 13:29:45'),
 'RET_LOT_LAXEMS_Last_File_Index': 29,
 'RET_LOT_LGBITS_Last_Date': Timestamp('2025-10-14 15:59:00'),
 'RET_LOT_LGBPST_Last_Date': Timestamp('2025-08-30 08:54:00'),
 'RET_LOT_LAXAPM_Last_Date': Timestamp('2025-12-08 09:00:00'),
 'PU_LOT_SAVCMP_Last_Date': Timestamp('2025-07-10 11:27:00'),
 'RET_LOT_SAVCMP_Last_Date': Timestamp('2025-07-21 09:06:00'),
 'RET_LOT_LGBPST_Last_File_Index': 0,
 'PU_LOT_OAKSTE_Last_Date': Timestamp('2025-10-30 07:01:00'),
 'RET_LOT_OAKSTE_Last_Date': Timestamp('2025-11-12 14:00:00'),
 'RET_LOT_LAXAPM_